# Assigment 3

In [ ]:
import igl
import numpy as np
import meshplot as mp
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve

In [ ]:
v, f = igl.read_triangle_mesh("data/bunny.off")
mp.plot(v, f)

# Vertex normal

In [ ]:
#Standard face normal
n_standard = igl.per_vertex_normals(v, f)
mp.plot(v, f, n=n_standard, shading={"flat": False})

In [ ]:
#Area-weighted face normal

n = igl.per_face_normals(v, f, np.ones(f.shape[0]))  

# Compute area - https://stackoverflow.com/questions/71346322/numpy-area-of-triangle-and-equation-of-a-plane-on-which-triangle-lies-on
face_areas = np.linalg.norm(n, axis=1) / 2  

n_area_weighted = np.zeros_like(v)
for i in range(f.shape[0]): # each face
    for j in range(3): # each vertex in face
        n_area_weighted[f[i, j]] += face_areas[i] * n[i]

mp.plot(v, f, n=n_area_weighted, shading={"flat": False})

# Curvature

In [24]:
#gaussian curvature
v,f  = igl.read_triangle_mesh("data/bunny.off")
G = igl.gaussian_curvature(v,f) 

# Plot the mesh colored by Gaussian curvature
p = mp.plot(v, f, G, shading={"wireframe": False})



Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…

In [ ]:
# mean
l = igl.cotmatrix(v, f)
m = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_VORONOI)

minv = sp.diags(1 / m.diagonal())

hn = -minv.dot(l.dot(v))
h = np.linalg.norm(hn, axis=1)
mp.plot(v, f, h)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…

In [ ]:
# principal curvature
v1, v2, k1, k2 = igl.principal_curvature(v, f)
h2 = 0.5 * (k1 + k2)
p = mp.plot(v, f, h2, shading={"wireframe": False}, return_plot=True)

avg = igl.avg_edge_length(v, f) / 2.0
p.add_lines(v + v1 * avg, v - v1 * avg, shading={"line_color": "red"})
p.add_lines(v + v2 * avg, v - v2 * avg, shading={"line_color": "green"})

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…

2

# Smoothing with the Laplacian

In [22]:
# explicit

def explicit_laplacian(v, f, _lambda, iter):
    l = igl.cotmatrix(v, f)  
    m = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_BARYCENTRIC)  
    minv = sp.diags(1 / m.diagonal()) # inverse

    vs = [v.copy()] 

    for _ in range(iter):
        v = v + _lambda * minv @ (l @ v)
        vs.append(v)

    return vs


v, f = igl.read_triangle_mesh("data/cow.off")
vs = explicit_laplacian(v, f, _lambda=0.000001 , iter=1000)


print(len(vs))
p = mp.plot(vs[1000], f, shading={"wireframe": True})



1001


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.002776…

In [23]:
# implicit
def implicit_laplacian(v, f, _lambda, iter):
    l = igl.cotmatrix(v, f)
    vs = [v.copy()]
    m = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_BARYCENTRIC)
    for _ in range(iter):
        s = (m - _lambda  * l)
        b = m.dot(v)
        k = spsolve(s, b)
        vs.append(k)

    return vs

v, f = igl.read_triangle_mesh("data/cow.off")
vs = implicit_laplacian(v, f, _lambda = 0.01, iter = 1)

print(len(vs))
p = mp.plot(vs[1], f, shading={"wireframe": True})




2


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.029718…